In [1]:
from pathlib import Path
import sys

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "medical_potential" / "config.py").is_file()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from medical_potential.config import BQ_DATASET_ID, MOA_INNOVATION_TABLE, PROJECT_ID
from medical_potential.gcp_utils import get_bq_client

bq_client = get_bq_client()
moa_innovation_table_id = f"{PROJECT_ID}.{BQ_DATASET_ID}.{MOA_INNOVATION_TABLE}"

moa_innovation_query = f"SELECT * FROM `{moa_innovation_table_id}`"
moa_innovation_df = bq_client.query(moa_innovation_query).to_dataframe()

moa_innovation_df.head()

c:\Users\P90022569\.conda\envs\cognito\Lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.11.15) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\P90022569\.conda\envs\cognito\Lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.11.15) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)


,drug_name,indication,mechanism_statement,moa_classification,score,guardrail,confidence_score,confidence_tier,justification_mechanism_summary,justification_novelty_vs_soc,...,justification_why_not_higher_score,grounding_urls,grounding_urls_by_step,narrative_rationale,sources_primary,sources_secondary,sources_tertiary,analysis_date,processing_time_seconds,created_at
0,Bgm-0504,"Obesity, Overweight",Gastric Inhibitory Polypeptide Receptor Agonis...,Me-too,3,PASS,0.95,Tier 1,Bgm-0504 is a dual agonist targeting the Gastr...,The mechanism is not novel to the indication o...,...,A score of 4 (Best-in-Class) was not awarded b...,[https://allsci.com/news/clinical-trials/brigh...,"{""step1_moa_search"":[],""step1_primary_indicati...",Bgm-0504 achieves a MoA Innovation score of 3 ...,"[""Yuan, J., et al. (2024). Molecular dynamics-...","[""BrightGene Bio-Medical Technology Phase III ...","[""FDA Orange Book (Zepbound Approval 2023)""]",2026-11-23,206.5,2026-11-23 10:05:06.455760+00:00


## MoA evals: faithfulness, precision and relevance

| Metric | What it measures | Module | Method |
|---|---|---|---|
| **Claim precision** (faithfulness) | Claims supported by the sources the run retrieved, divided by ALL extracted claims. Unverifiable and errored claims count against the score. | `claim_verification` | Claude fetches the saved source pages |
| **Judge faithfulness** | Is the assigned score supported by its own justification, and is it internally consistent | `llm_judge` | LLM judge against the rubric |
| **Context precision** | Share of the retrieved sources that are relevant to this drug, mechanism and indication (also per step) | `context_precision` | LLM judge over saved sources |
| **Field relevance** | Does each `justification_*` field answer its own question for this drug (1-5) | `relevance_judge` | LLM judge, per-field rubric |
| **Claim relevance** | Share of claims that bear on the classification (direct = 1, contextual = 0.5, irrelevant = 0) | `relevance_judge` | LLM judge |

The composite is a weighted average of the components that ran (`scoring.DEFAULT_WEIGHTS`). Components that failed or were skipped are dropped and the weights renormalised; `composite_coverage` shows how much weight was used, and no status is given below `MIN_COVERAGE`.

Retrieved context is available: every MoA run saves its grounding sources to `moa_innovation/sources/<drug>/sources_*.json`, and the evals read it from there. The run must have happened on the same filesystem, otherwise claim verification and context precision report errors.

In [2]:
from medical_potential.config import CLAUDE_SONNET_MODEL
from medical_potential.claude_utils import get_claude_call_stats, reset_claude_call_stats
from medical_potential.moa_innovation.evals.bq_push import build_bq_rows, push_eval_rows
from medical_potential.moa_innovation.evals.claim_verification import (
    run_claim_verification,
    summarize_claims,
)
from medical_potential.moa_innovation.evals.classification_check import (
    check_classification_score,
    summarize_classification_check,
)
from medical_potential.moa_innovation.evals.context_precision import run_context_precision
from medical_potential.moa_innovation.evals.llm_judge import run_llm_judge
from medical_potential.moa_innovation.evals.relevance_judge import (
    run_claim_relevance,
    run_field_relevance,
    summarize_claim_relevance,
)
from medical_potential.moa_innovation.evals.scoring import DEFAULT_WEIGHTS, build_eval_scores
from medical_potential.moa_innovation.evals.utils import eval_timestamp, latest_rows

SELECTED_MOLECULES = [
 'Bgm-0504'
]
EVAL_MODEL = CLAUDE_SONNET_MODEL  # Claude on Vertex AI
EVAL_RUN_ID = eval_timestamp()
PUSH_TO_BQ = False

FETCH_SOURCE_PAGES_FOR_PRECISION = False  # True lets Claude open every source (slower, costlier)
SCORE_WEIGHTS = dict(DEFAULT_WEIGHTS)     # edit to re-weight; missing components are renormalised
MIN_COVERAGE = 0.5

print(f"Loaded {len(moa_innovation_df):,} source rows")
print(f"Evaluation run: {EVAL_RUN_ID}")
print(f"Weights: {SCORE_WEIGHTS}")

Loaded 1 source rows
Evaluation run: 2026-11-23T10:33:00.886691+00:00
Weights: {'claim_precision': 0.4, 'judge_faithfulness': 0.1, 'context_precision': 0.15, 'field_relevance': 0.2, 'claim_relevance': 0.15}


In [3]:
classification_results = check_classification_score(moa_innovation_df)
classification_summary = summarize_classification_check(classification_results)

print(classification_summary)
classification_failures = classification_results[
    ~classification_results["classification_check_pass"]
]
classification_failures[[
    "source_key",
    "source_classification",
    "source_score",
    "expected_score",
    "classification_check_reason",
]]

{'total_rows': 1, 'passed_rows': 1, 'failed_rows': 0, 'pass_rate': 1.0, 'failures_by_class': {}, 'score_crosstab': {3: {'Me-too': 1}}}


,source_key,source_classification,source_score,expected_score,classification_check_reason


In [4]:
selected_rows = latest_rows(moa_innovation_df, SELECTED_MOLECULES)
selected_rows[["drug_name", "created_at", "moa_classification", "score"]]

,drug_name,created_at,moa_classification,score
0,Bgm-0504,2026-11-23 10:05:06.455760+00:00,Me-too,3


In [5]:
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

FRAME_KEYS = (
    "claim_rows", "claim_results", "judge_results", "context_precision_results",
    "field_relevance_results", "claim_relevance_rows", "claim_relevance_results",
)


def _stage(name, molecule_name, errors, function):
    """Run one eval stage; a failure is recorded and does not stop the other stages."""
    try:
        return function()
    except Exception as error:
        errors.append(f"{name}: {error}")
        print(f"Stage '{name}' failed for {molecule_name}: {error}")
        return pd.DataFrame()


def evaluate_molecule(selected_row):
    molecule_name = str(selected_row["drug_name"])
    molecule_rows = selected_row.to_frame().T
    reset_claude_call_stats()
    started_at = time.perf_counter()
    errors = []
    frames = {key: pd.DataFrame() for key in FRAME_KEYS}

    # Faithfulness / claim precision
    claim_rows = _stage("claim_verification", molecule_name, errors,
                        lambda: run_claim_verification(molecule_rows, model=EVAL_MODEL))
    frames["claim_rows"] = claim_rows
    frames["claim_results"] = summarize_claims(claim_rows)
    frames["judge_results"] = _stage("llm_judge", molecule_name, errors,
                                     lambda: run_llm_judge(molecule_rows, model=EVAL_MODEL))

    # Precision of the retrieved context
    frames["context_precision_results"] = _stage(
        "context_precision", molecule_name, errors,
        lambda: run_context_precision(molecule_rows, model=EVAL_MODEL, fetch_pages=FETCH_SOURCE_PAGES_FOR_PRECISION))

    # Relevance
    frames["field_relevance_results"] = _stage("field_relevance", molecule_name, errors,
                                               lambda: run_field_relevance(molecule_rows, model=EVAL_MODEL))
    frames["claim_relevance_rows"] = _stage("claim_relevance", molecule_name, errors,
                                            lambda: run_claim_relevance(claim_rows, molecule_rows, model=EVAL_MODEL))
    frames["claim_relevance_results"] = summarize_claim_relevance(frames["claim_relevance_rows"])

    print(
        f"{molecule_name}: {'failed stages: ' + '; '.join(errors) if errors else 'all stages ok'} "
        f"in {time.perf_counter() - started_at:.1f}s, usage={get_claude_call_stats()}"
    )
    return frames


collected = {key: [] for key in FRAME_KEYS}

with ThreadPoolExecutor(max_workers=1) as executor:  # raise only if your Claude quota allows parallel molecules
    futures = [executor.submit(evaluate_molecule, row) for _, row in selected_rows.iterrows()]
    for future in as_completed(futures):
        for key, frame in future.result().items():
            if not frame.empty:
                collected[key].append(frame)

(claim_rows, claim_results, judge_results, context_precision_results,
 field_relevance_results, claim_relevance_rows, claim_relevance_results) = (
    pd.concat(collected[key], ignore_index=True) if collected[key] else pd.DataFrame() for key in FRAME_KEYS
)

print(f"Completed evaluations for {len(selected_rows)} molecule(s)")

[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/anthropic/models/claude-sonnet-4-6:streamRawPredict "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/anthropic/models/claude-sonnet-4-6:streamRawPredict "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/anthropic/models/claude-sonnet-4-6:streamRawPredict "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/anthropic/models/claude-sonnet-4-6:streamRawPredict "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/anthropic/models/claude-sonnet-4-6:streamRawPredict "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://aiplatform.googleapis

Bgm-0504: all stages ok in 314.3s, usage=[{'model_used': 'claude-sonnet-4-6', 'input_token_count': 326396, 'output_token_count': 17423, 'cache_creation_input_token_count': 0, 'cache_read_input_token_count': 0, 'total_token_count': 343819, 'total_num_of_calls': 20, 'num_of_search_enabled_calls': 15, 'num_of_search_queries': 0}]
Completed evaluations for 1 molecule(s)


In [6]:
eval_results = build_eval_scores(
    classification_results,
    claim_results,
    judge_results,
    context_precision_results,
    field_relevance_results,
    claim_relevance_results,
    amber_threshold=65.0,
    green_threshold=85.0,
    weights=SCORE_WEIGHTS,
    min_coverage=MIN_COVERAGE,
)

summary_columns = [
    "source_key",
    "source_classification",
    "source_score",
    "classification_check_pass",
    "claim_precision_score",
    "judge_faithfulness_score",
    "context_precision_score",
    "field_relevance_score",
    "claim_relevance_score",
    "composite_coverage",
    "composite_score",
    "eval_status",
]
eval_results[[column for column in summary_columns if column in eval_results.columns]]

,source_key,source_classification,source_score,classification_check_pass,claim_precision_score,judge_faithfulness_score,context_precision_score,field_relevance_score,claim_relevance_score,composite_coverage,composite_score,eval_status
0,Bgm-0504|2026-11-23 10:05:06.455760+00:00,Me-too,3,True,0.809524,1.0,0.971429,0.875,0.642857,1.0,84.095238,Amber


In [7]:
# Drill-down: where precision and relevance are lost

# 1. Claims that are not supported by the retrieved sources (hallucination risk)
if not claim_rows.empty:
    display_columns = [c for c in ["field", "claim", "verdict", "evidence"] if c in claim_rows.columns]
    display(claim_rows.loc[claim_rows["verdict"] != "supported", display_columns])

# 2. Precision of the retrieved sources, per MoA step
if not context_precision_results.empty and "context_precision_by_step" in context_precision_results.columns:
    for _, result_row in context_precision_results.iterrows():
        print(result_row["drug_name"], "overall:", result_row.get("context_precision_score"))
        by_step = result_row.get("context_precision_by_step")
        if isinstance(by_step, dict):
            display(pd.DataFrame(by_step).T)
        details = result_row.get("context_details")
        if isinstance(details, list):
            display(pd.DataFrame(details).query("verdict != 'relevant'")[["title", "domain", "steps", "verdict", "reason"]])

# 3. Relevance per justification field
relevance_columns = [c for c in field_relevance_results.columns if c.startswith("relevance_") and c != "relevance_details"]
if relevance_columns:
    display(field_relevance_results[["drug_name", *relevance_columns, "field_relevance_mean", "fields_not_drug_specific"]])

# 4. Claims that do not bear on the classification
if not claim_relevance_rows.empty:
    display(claim_relevance_rows.loc[claim_relevance_rows["claim_relevance"] != "direct", ["field", "claim", "claim_relevance"]])

,field,claim,verdict,evidence
7,justification_novelty_vs_soc,Tirzepatide is currently a first-line standard...,unverifiable,The Frontiers in Endocrinology (2024) review c...
10,justification_competitor_comparison,Tirzepatide has demonstrated approximately 21%...,unverifiable,None of the successfully fetched sources direc...
11,justification_competitor_comparison,"Retatrutide is a triple agonist targeting GIP,...",unverifiable,None of the successfully fetched sources descr...
12,justification_competitor_comparison,Retatrutide has demonstrated weight loss appro...,unverifiable,None of the successfully fetched sources direc...


Bgm-0504 overall: 0.9714285714285714


,total,relevant,precision
step4_biological_rationale,13.0,13.0,1.000000
step6_mechanism_failures,6.0,6.0,1.000000
step7_mechanistic_improvement,22.0,22.0,1.000000
step5_clinical_validation,6.0,5.0,0.833333
step8_mechanism_currency,1.0,1.0,1.000000


,title,domain,steps,verdict,reason
13,youtube.com,youtube.com,[step5_clinical_validation],cannot_tell,YouTube video used in clinical validation step...


,drug_name,relevance_mechanism_summary,relevance_novelty_vs_soc,relevance_competitor_comparison,relevance_biological_rationale,relevance_prior_validation_failure,relevance_why_not_higher_score,field_relevance_mean,fields_not_drug_specific
0,Bgm-0504,5,5,5,3,4,5,4.5,1


,field,claim,claim_relevance
1,justification_mechanism_summary,GLP-1R agonism produces central appetite suppr...,contextual
2,justification_mechanism_summary,GLP-1R agonism produces delayed gastric emptying.,contextual
3,justification_mechanism_summary,GIPR agonism improves insulin sensitivity.,contextual
4,justification_mechanism_summary,GIPR agonism contributes to lipid buffering in...,contextual
5,justification_novelty_vs_soc,Tirzepatide is a dual GIP/GLP-1 receptor agonist.,contextual
6,justification_novelty_vs_soc,Tirzepatide was approved in 2023 for obesity u...,contextual
7,justification_novelty_vs_soc,Tirzepatide is currently a first-line standard...,contextual
8,justification_competitor_comparison,Semaglutide is a GLP-1 mono-agonist used for o...,contextual
10,justification_competitor_comparison,Tirzepatide has demonstrated approximately 21%...,contextual
11,justification_competitor_comparison,"Retatrutide is a triple agonist targeting GIP,...",contextual


In [ ]:
import pandas as pd

output_dir = repo_root / "medical_potential" / "moa_innovation" / "evals"
excel_output_path = output_dir / f"moa_evaluation_results_{pd.Timestamp.now():%Y%m%d_%H%M%S}.xlsx"


def _excel_safe(dataframe):
    """Excel cannot store lists/dicts or tz-aware datetimes."""
    export_dataframe = dataframe.copy()
    if export_dataframe.columns.duplicated().any():
        duplicates = sorted(set(export_dataframe.columns[export_dataframe.columns.duplicated()]))
        raise ValueError(f"Duplicate column names before Excel export: {duplicates}")
    for column in export_dataframe.select_dtypes(include=["datetimetz"]).columns:
        export_dataframe[column] = export_dataframe[column].dt.tz_localize(None)
    for column in export_dataframe.columns:
        if export_dataframe[column].map(lambda value: isinstance(value, (list, dict))).any():
            export_dataframe[column] = export_dataframe[column].map(
                lambda value: json.dumps(value, default=str) if isinstance(value, (list, dict)) else value
            )
    return export_dataframe


import json


def _join(parts):
    return " | ".join(part for part in parts if part)


def _field_rationales(details):
    entries = (details or {}).get("fields", []) if isinstance(details, dict) else []
    return _join(
        f"{e.get('field', '').removeprefix('justification_')}: {e.get('score')}/5 - {e.get('rationale', '')}"
        for e in entries if isinstance(e, dict)
    )


def build_eval_metrics(overall, judge, context, field_rel, claim_rel_rows):
    """One row per molecule; every score is 0-1."""
    records = []
    for _, row in overall.iterrows():
        key = row["source_key"]

        def pick(frame):
            if frame is None or frame.empty or "source_key" not in frame:
                return {}
            match = frame[frame["source_key"] == key]
            return match.iloc[0].to_dict() if len(match) else {}

        j, c, f = pick(judge), pick(context), pick(field_rel)

        # Faithfulness: LLM judge, overall rating 1-5 rescaled to 0-1.
        faith = row.get("judge_faithfulness_score")
        faith_rationale = _join([
            j.get("faithfulness_score_rationale"),
            f"Score supported: {j.get('score_supported')} - {j.get('score_supported_rationale')}" if j else "",
            f"Why not higher valid: {j.get('why_not_higher_valid')} - {j.get('why_not_higher_valid_rationale')}" if j else "",
            f"Internal consistency: {j.get('internal_consistency')} - {j.get('internal_consistency_rationale')}" if j else "",
        ])

        # Relevance: mean of field relevance and claim relevance (whichever are available).
        field_score, claim_score = row.get("field_relevance_score"), row.get("claim_relevance_score")
        parts = [v for v in (field_score, claim_score) if pd.notna(v)]
        relevance = sum(parts) / len(parts) if parts else None
        claim_text = ""
        if claim_rel_rows is not None and not claim_rel_rows.empty:
            labels = claim_rel_rows.loc[claim_rel_rows["source_key"] == key, "claim_relevance"].value_counts()
            claim_text = (
                f"Claims: {labels.get('direct', 0)} direct, {labels.get('contextual', 0)} contextual, "
                f"{labels.get('irrelevant', 0)} irrelevant (score {claim_score:.2f})" if pd.notna(claim_score) else ""
            )
        relevance_rationale = _join([
            f"Field relevance {field_score:.2f}: " + _field_rationales(f.get("relevance_details")) if pd.notna(field_score) else "",
            claim_text,
        ])

        # Precision: share of retrieved sources judged relevant.
        precision = row.get("context_precision_score")
        irrelevant = [
            f"{d.get('title') or d.get('url')}: {d.get('reason')}"
            for d in (c.get("context_details") or []) if isinstance(d, dict) and d.get("verdict") != "relevant"
        ]
        precision_rationale = _join([
            f"{c.get('sources_relevant')} of {c.get('sources_total')} retrieved sources relevant"
            f" ({c.get('sources_irrelevant')} irrelevant, {c.get('sources_cannot_tell')} cannot tell)" if c.get("sources_total") else c.get("context_error", ""),
            "Not relevant: " + "; ".join(irrelevant[:10]) if irrelevant else "",
        ])

        records.append({
            "Drug Name": row.get("drug_name") or row.get("source_key", "").split("|")[0],
            "Faithfulness Score": faith,
            "Faithfulness Rationale": faith_rationale,
            "Relevance Score": relevance,
            "Relevance Rationale": relevance_rationale,
            "Precision Score": precision,
            "Precision Rationale": precision_rationale,
        })
    return pd.DataFrame(records)


eval_metrics = build_eval_metrics(
    eval_results, judge_results, context_precision_results, field_relevance_results, claim_relevance_rows
)

with pd.ExcelWriter(excel_output_path, engine="openpyxl") as writer:
    for dataframe, sheet_name in [
        (classification_results, "classification_results"),
        (claim_rows, "claim_results"),
        (judge_results, "judge_results"),
        (context_precision_results, "context_precision"),
        (field_relevance_results, "field_relevance"),
        (claim_relevance_rows, "claim_relevance"),
        (eval_results, "overall_results"),
        (eval_metrics, "eval metrics"),
    ]:
        if dataframe is not None and not dataframe.empty:
            _excel_safe(dataframe).to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved evaluation results to {excel_output_path}")

if PUSH_TO_BQ:
    push_eval_rows(build_bq_rows(eval_results.to_dict("records"), EVAL_RUN_ID, EVAL_MODEL))
    print("Pushed eval rows to BigQuery")